In [283]:
import random

In [284]:
#R.<x> = GF(2)[]
F = GF(256)
F

Finite Field in z8 of size 2^8

In [285]:
m = 255 # codeword size
n = 223 # word size
t = 15 # can fix t errors

In [286]:
distance = (m - n + 1)/m
distance

11/85

In [287]:
# define x as a field GF(256) element
R.<x> = PolynomialRing(F)

def print_list_hex(p):
    arr = [c.to_integer() for c in p]
    print(bytearray(arr).hex())

# sample random message -> a deg=n polynomial in GF(256).
word = sum(F.random_element() * x^i for i in range(n))
print_list_hex(word.list())

4dfe92bc42e69dbb8a9b6ed1753f2009c7e80d07d48a8b7bd0440de0649365f6e3010149fde98ceafcf1a7eba0b1281c478968bb4eacb0c370790b8dc28f31808fe6ad9f763bf722ddfc1433fdaafa251b74a4a408d20d18c216e60f8b12d3b90fe2b2f2e546ede7123c54c4b9b2cd0dfe4ea85c1c965e48ec8b7c7d32311399519e49e146d1cb1af773da317f466c22118588e93cf34907967835059bf914499f29f3ea5dc7e039c25f22b563af1f81b6420e1e04cc638ce5bc4d724393d8e06ab28c00e2f26de95abf06a354114d75f562b54308185c387964c31d38b3a3


In [288]:

# All non zero elements in F. Order(F) - 1 = 255
f = [f_i for f_i in F if f_i != 0]

assert len(f) == m

codeword = [word(f_i) for f_i in f]

assert len(codeword) == m
print_list_hex(codeword)

# Uncomment to add errors in codeword
# for i in range(t):
  #  codeword[i] = F.random_element()

65babc936d36142e3b6b59792bd3e9e7303a364f6ff1e6d46b0b0eb50942bcb2ae17b39f95d35ef81d43a8ba344360e5b047e221fcc2f208c87d7a6f3f055150e6a9e967e4cd5be3520a0978877284b7e5b057368cd016e418c68fb64915cf19b1786efb49c81cc1a258b6ba03c765327e7645264e9d0465eeb737fd3c11f747fddb5c7c34bf62fa778d63f5c1a22002aee13c1100ab9216b24c0ed50a31a8ba87e46a20d21bbcabb35e853ea0816c23231cbe32aec4446c37cb2c55beb6cbd459a824a90729ebe30e6304597a66a9ca32bb3bc4cb81b07bd7734acf96b6406819d827c2c270b6de3009b02591514f453f46e99a54f2cb4464a20e3b705334


In [289]:
# Solve C(x_i) = b_i * E(x_i) with Berlekamp-Welch

# F = GF(2^8, 'a', modulus=x^8 + x^4 + x^3 + x + 1)

# codewords is a list of F elements 
# f is a list of non-zero F elements

deg_C = 237 # n - 1 + t
deg_E = 15 # t

M = Matrix(F, m, deg_C + deg_E + 1)

for i, (f_i, b_i) in enumerate(zip(f, codeword)):
    # Fill in coefficients for C(x)
    for j in range(deg_C + 1):
        M[i, j] = f_i^j
    # Fill in coefficients for E(x), multiplied by -b_i
    for j in range(deg_E):
        M[i, deg_C + 1 + j] = -b_i * (f_i^j)


rhs = vector(F, [b_i * (f_i**deg_E) for f_i, b_i in zip(f, codeword)])
# rhs = vector(F, m)  # Right-hand side is the zero vector

# Solve the linear system over F
solution = M.solve_right(rhs)
assert len(solution) == deg_C + deg_E + 1

In [292]:
C = sum(solution[i] * x^i for i in range(deg_C + 1))
E = sum(solution[deg_C + 1 + i] * x^i for i in range(deg_E))
E += x ^ deg_E
G = C/E

In [293]:
assert G.numerator() == word
print_list_hex(G.numerator().list())

4dfe92bc42e69dbb8a9b6ed1753f2009c7e80d07d48a8b7bd0440de0649365f6e3010149fde98ceafcf1a7eba0b1281c478968bb4eacb0c370790b8dc28f31808fe6ad9f763bf722ddfc1433fdaafa251b74a4a408d20d18c216e60f8b12d3b90fe2b2f2e546ede7123c54c4b9b2cd0dfe4ea85c1c965e48ec8b7c7d32311399519e49e146d1cb1af773da317f466c22118588e93cf34907967835059bf914499f29f3ea5dc7e039c25f22b563af1f81b6420e1e04cc638ce5bc4d724393d8e06ab28c00e2f26de95abf06a354114d75f562b54308185c387964c31d38b3a3
